# Resume Cleaning & Preprocessing

In [ ]:
import re
import html
import json
import unicodedata
import warnings
import sys
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
from project_paths import CLEANED_DIR, RESUME_CSV, rel

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

CONFIG = {
    "resume_path": RESUME_CSV,
    "out_root": CLEANED_DIR,
    "min_words": 50,
    "min_chars_keep": 50,
}

for d in ["master", "normalized", "analysis", "quarantine", "reports"]:
    (CONFIG["out_root"] / d).mkdir(parents=True, exist_ok=True)


def step(msg, df=None):
    if df is not None:
        print(f"[STEP] {msg:<48} rows={len(df):>7,}  cols={df.shape[1]}")
    else:
        print(f"[STEP] {msg}")


# text cleaning
URL_RE = re.compile(r"http\S+|www\.\S+", re.IGNORECASE)
EMAIL_RE = re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b")
PHONE_RE = re.compile(r"(?:\+?\d{1,3}[-.\s]?)?(?:\(?\d{3}\)?[-.\s]?)\d{3}[-.\s]?\d{4}")
HTML_RE = re.compile(r"<[^>]+>")
MULTISPACE_RE = re.compile(r"\s+")


def light_clean_text(text, drop_contact=False):
    """Light, meaning-preserving clean: keeps case + punctuation."""
    if not isinstance(text, str) or not text.strip():
        return np.nan
    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text).replace("\ufffd", " ")
    text = HTML_RE.sub(" ", text)
    if drop_contact:
        text = URL_RE.sub(" ", text)
        text = EMAIL_RE.sub(" ", text)
        text = PHONE_RE.sub(" ", text)
    text = text.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    text = MULTISPACE_RE.sub(" ", text).strip()
    return text if text else np.nan


def norm_category(v):
    """'INFORMATION-TECHNOLOGY' -> 'information_technology'."""
    if not isinstance(v, str) or not v.strip():
        return np.nan
    return re.sub(r"[^a-z0-9]+", "_", v.lower()).strip("_") or np.nan


print("Resume input:", rel(CONFIG["resume_path"]))
print("demo clean ->", light_clean_text("  John   has  C++/.NET   skills  a@b.com ", drop_contact=True))

## 1. Load and profile the resume dataset

In [ ]:
resumes = pd.read_csv(CONFIG["resume_path"], low_memory=False)
AUDIT = {"row_counts": {"raw_resumes": int(len(resumes))}, "issues": {}, "scores": {}, "notes": []}

step("loaded Resume.csv", resumes)
print("columns:", list(resumes.columns))
print("\nCategory distribution (Role applied):")
print(resumes["Category"].value_counts().to_string())
print("\nResume_str length (chars): min %d | median %d | max %d"
      % (resumes["Resume_str"].str.len().min(),
         resumes["Resume_str"].str.len().median(),
         resumes["Resume_str"].str.len().max()))

## 2. Clean text, extract contacts, normalize category

In [ ]:
df = resumes.rename(columns={"ID": "candidate_id",
                             "Resume_str": "resume_raw",
                             "Category": "category"}).copy()

def first_match(rx, text):
    if not isinstance(text, str):
        return np.nan
    m = rx.search(text)
    return m.group(0) if m else np.nan

# extract contact info first
df["candidate_email"] = df["resume_raw"].map(lambda t: first_match(EMAIL_RE, t))
df["candidate_phone"] = df["resume_raw"].map(lambda t: first_match(PHONE_RE, t))

# contacts removed for privacy
df["resume_clean"] = df["resume_raw"].map(lambda t: light_clean_text(t, drop_contact=True))

# category -> normalized role label
df["category_clean"] = df["category"].map(norm_category)
df["role_applied"] = df["category_clean"]

# length features
df["resume_char_count"] = df["resume_clean"].fillna("").str.len()
df["resume_word_count"] = df["resume_clean"].fillna("").str.split().str.len()

AUDIT["issues"]["emails_found"] = int(df["candidate_email"].notna().sum())
AUDIT["issues"]["phones_found"] = int(df["candidate_phone"].notna().sum())
step("cleaned + contacts extracted", df)
print("emails:", AUDIT['issues']['emails_found'], "| phones:", AUDIT['issues']['phones_found'])
print("word count: min %d | median %d | max %d"
      % (df["resume_word_count"].min(), df["resume_word_count"].median(), df["resume_word_count"].max()))

## 3. Extract skills, education level, and years of experience

In [ ]:
SKILL_DICT = {
    # --- programming / data ---
    "python": r"python", "java": r"java(?!script)", "javascript": r"java\s?script|\bjs\b",
    "c++": r"c\+\+", "c#": r"c#|c sharp", ".net": r"\.net|dotnet|asp\.net",
    "sql": r"\bsql\b|mysql|postgre|t-?sql|pl/?sql", "sql_server": r"sql server|mssql",
    "r_language": r"\br programming\b|\brstudio\b", "scala": r"\bscala\b", "go_language": r"\bgolang\b",
    "html": r"\bhtml5?\b", "css": r"\bcss3?\b", "php": r"\bphp\b", "ruby": r"\bruby\b",
    "matlab": r"matlab", "sas": r"\bsas\b", "vba": r"\bvba\b", "perl": r"\bperl\b",
    # --- data / ml / bi ---
    "machine_learning": r"machine learning|\bml\b", "deep_learning": r"deep learning",
    "nlp": r"natural language processing|\bnlp\b", "data_analysis": r"data analysis|data analytics",
    "data_science": r"data science", "tableau": r"tableau", "power_bi": r"power\s?bi",
    "excel": r"\bexcel\b|ms excel|microsoft excel", "spark": r"\bspark\b|pyspark",
    "hadoop": r"hadoop", "tensorflow": r"tensorflow", "pytorch": r"pytorch", "pandas": r"pandas",
    # --- cloud / devops ---
    "aws": r"\baws\b|amazon web services", "azure": r"\bazure\b", "gcp": r"\bgcp\b|google cloud",
    "docker": r"docker", "kubernetes": r"kubernetes|k8s", "linux": r"\blinux\b|unix",
    "git": r"\bgit\b|github|gitlab", "jenkins": r"jenkins", "ci_cd": r"ci/?cd",
    # --- business / office / pm ---
    "project_management": r"project management", "agile": r"\bagile\b|scrum",
    "ms_office": r"ms office|microsoft office", "powerpoint": r"power\s?point",
    "communication": r"communication skills?", "leadership": r"leadership",
    "customer_service": r"customer service", "sales": r"\bsales\b", "marketing": r"marketing",
    "negotiation": r"negotiation", "budgeting": r"budgeting|budget management",
    # --- finance / accounting ---
    "accounting": r"accounting", "quickbooks": r"quickbooks", "auditing": r"auditing|audit",
    "financial_analysis": r"financial analysis", "taxation": r"\btax(ation)?\b", "sap": r"\bsap\b",
    "payroll": r"payroll", "forecasting": r"forecasting",
    # --- hr / healthcare / other domains ---
    "recruitment": r"recruitment|recruiting|talent acquisition", "onboarding": r"onboarding",
    "patient_care": r"patient care", "nursing": r"\bnursing\b|registered nurse|\brn\b",
    "autocad": r"autocad", "solidworks": r"solidworks", "photoshop": r"photoshop",
    "illustrator": r"illustrator", "adobe": r"adobe", "seo": r"\bseo\b",
    "social_media": r"social media", "salesforce": r"salesforce", "crm": r"\bcrm\b",
}
SKILL_RX = {canon: re.compile(pat, re.IGNORECASE) for canon, pat in SKILL_DICT.items()}


def extract_skills(text):
    if not isinstance(text, str):
        return []
    return sorted([canon for canon, rx in SKILL_RX.items() if rx.search(text)])



EDU_PATTERNS = {
    "doctorate": r"ph\.?\s?d|doctorate|doctoral",
    "master": r"master'?s|m\.?\s?sc|m\.?\s?b\.?a|m\.?\s?tech|\bm\.?a\b|post[- ]?graduate",
    "bachelor": r"bachelor'?s|b\.?\s?sc|b\.?\s?tech|\bb\.?e\b|\bb\.?a\b|b\.?com|under[- ]?graduate",
    "associate": r"associate'?s degree|\ba\.?a\.?s\b",
    "diploma": r"\bdiploma\b",
    "high_school": r"high school|secondary school|\bg\.?e\.?d\b",
}
EDU_RANK = {"high_school": 1, "diploma": 2, "associate": 3, "bachelor": 4, "master": 5, "doctorate": 6}
EDU_RX = {lvl: re.compile(pat, re.IGNORECASE) for lvl, pat in EDU_PATTERNS.items()}
YEARS_RX = re.compile(r"(\d{1,2})\+?\s*(?:\+)?\s*years?", re.IGNORECASE)


def extract_education(text):
    if not isinstance(text, str):
        return []
    return sorted([lvl for lvl, rx in EDU_RX.items() if rx.search(text)], key=lambda x: EDU_RANK[x])


def highest_education(levels):
    return max(levels, key=lambda x: EDU_RANK[x]) if levels else np.nan


def extract_years(text):
    if not isinstance(text, str):
        return np.nan
    vals = [int(y) for y in YEARS_RX.findall(text) if y.isdigit()]
    vals = [v for v in vals if 0 < v <= 50] 
    return max(vals) if vals else np.nan


df["extracted_skills"] = df["resume_clean"].map(extract_skills)
df["skill_count"] = df["extracted_skills"].map(len)
df["education_levels"] = df["resume_clean"].map(extract_education)
df["highest_education"] = df["education_levels"].map(highest_education)
df["years_experience"] = df["resume_clean"].map(extract_years)

step("skills / education / experience extracted", df)
print("avg skills/resume:", round(df["skill_count"].mean(), 2),
      "| resumes w/ education:", int(df["highest_education"].notna().sum()),
      "| resumes w/ years:", int(df["years_experience"].notna().sum()))

## 4. Deduplicate and quality-check

In [ ]:
n0 = len(df)

# (1) exact duplicate resume text (keep first)
dup_mask = df["resume_clean"].duplicated(keep="first") & df["resume_clean"].notna()
AUDIT["issues"]["dup_resumes_removed"] = int(dup_mask.sum())
df = df[~dup_mask].reset_index(drop=True)

# (2) quarantine empty/unusable resumes (too short to be meaningful)
empty_mask = df["resume_clean"].isna() | (df["resume_char_count"] < CONFIG["min_chars_keep"])
quarantined = df[empty_mask].copy()
if len(quarantined):
    quarantined.to_csv(CONFIG["out_root"] / "quarantine" / "unusable_resumes.csv", index=False)
df = df[~empty_mask].reset_index(drop=True)
AUDIT["issues"]["resumes_quarantined"] = int(len(quarantined))

# (3) flag (do NOT delete) low-quality short resumes
df["is_low_quality"] = df["resume_word_count"] < CONFIG["min_words"]
AUDIT["issues"]["low_quality_flagged"] = int(df["is_low_quality"].sum())

step(f"after dedup+quarantine (removed {n0 - len(df):,})", df)
print("dup removed:", AUDIT["issues"]["dup_resumes_removed"],
      "| quarantined:", AUDIT["issues"]["resumes_quarantined"],
      "| low-quality flagged:", AUDIT["issues"]["low_quality_flagged"])

## 5. Build `resume_master`, validate, and save

In [ ]:
df["resume_completeness_score"] = (
    0.40 * (df["skill_count"] > 0)
    + 0.25 * df["highest_education"].notna()
    + 0.20 * (df["resume_word_count"] >= 100)
    + 0.15 * df["years_experience"].notna()
).round(3)

resume_master = df[[
    "candidate_id", "category", "category_clean", "role_applied",
    "resume_clean", "resume_char_count", "resume_word_count",
    "extracted_skills", "skill_count",
    "education_levels", "highest_education", "years_experience",
    "candidate_email", "candidate_phone",
    "resume_completeness_score", "is_low_quality",
]].copy()


assert resume_master["candidate_id"].is_unique, "candidate_id must be unique"
assert resume_master["resume_clean"].notna().all(), "resume_clean must be non-null after quarantine"
print("Validation OK: candidate_id unique, resume_clean non-null.")

cov = lambda c: round(float(resume_master[c].notna().mean()) * 100, 1)
metrics = {
    "n_resumes": int(len(resume_master)),
    "n_categories": int(resume_master["category"].nunique()),
    "skill_coverage_pct": round(float((resume_master["skill_count"] > 0).mean()) * 100, 1),
    "education_coverage_pct": cov("highest_education"),
    "experience_coverage_pct": cov("years_experience"),
    "avg_skills_per_resume": round(float(resume_master["skill_count"].mean()), 2),
    "avg_completeness": round(float(resume_master["resume_completeness_score"].mean()), 3),
}
scores = {
    "data_quality": int(round(100 - resume_master["is_low_quality"].mean() * 100)),
    "nlp_readiness": 100,  # text fully cleaned & non-null
    "skill_extraction_readiness": int(round(metrics["skill_coverage_pct"])),
    "matching_readiness": int(round(0.5 * metrics["skill_coverage_pct"]
                                    + 0.5 * 100 * (resume_master["resume_word_count"] >= 100).mean())),
}
AUDIT["scores"] = scores
AUDIT["metrics"] = metrics
AUDIT["notes"] = [
    "Name not extracted (dataset anonymized; reliable name detection needs NER).",
    "Contacts mostly scrubbed in source; extracted where present, then removed from text.",
    "Skills normalized to snake_case to match the job-side vocabulary for downstream matching.",
]
print("\nMetrics:", json.dumps(metrics, indent=2))
print("Scores :", json.dumps(scores, indent=2))

In [ ]:

def serialize_lists(df_in):
    df_in = df_in.copy()
    for c in df_in.columns:
        if df_in[c].map(lambda v: isinstance(v, list)).any():
            df_in[c] = df_in[c].map(lambda v: "|".join(map(str, v)) if isinstance(v, list) else "")
    return df_in


# skill frequency across resumes
all_skills = [s for lst in resume_master["extracted_skills"] if isinstance(lst, list) for s in lst]
skill_freq = (
    pd.Series(Counter(all_skills)).sort_values(ascending=False)
    .rename_axis("skill").reset_index(name="resume_count")
)
skill_freq.to_csv(CONFIG["out_root"] / "analysis" / "resume_skill_frequency.csv", index=False)
print("Top 10 resume skills:\n", skill_freq.head(10).to_string(index=False))

# category distribution (for SQL / dashboard)
(resume_master["category_clean"].value_counts()
 .rename_axis("role_applied").reset_index(name="resume_count")
 .to_csv(CONFIG["out_root"] / "analysis" / "resume_category_distribution.csv", index=False))

# save master + a normalized copy
serialize_lists(resume_master).to_csv(CONFIG["out_root"] / "master" / "resume_master.csv", index=False)
serialize_lists(resume_master).to_csv(CONFIG["out_root"] / "normalized" / "resume_clean.csv", index=False)

# audit report
AUDIT["row_counts"]["resume_master"] = int(len(resume_master))
with open(CONFIG["out_root"] / "reports" / "resume_quality_audit.json", "w", encoding="utf-8") as fh:
    json.dump(AUDIT, fh, indent=2, default=str)

print("\nSaved: master/resume_master.csv (", len(resume_master), "rows )")
print("DONE. Outputs in:", rel(CONFIG["out_root"]))